# Using tags on ensembles

Ensembles in Sumo can carry a list of `tags`. This notebook shows how to:

1. list the tags that are in use,
2. find ensembles by tag (single tag, any of several, all of several),
3. find ensembles with or without tags,
4. combine tags with other filters,
5. get from tagged ensembles to the data objects inside them.

In [ ]:
from fmu.sumo.explorer import Explorer

In [ ]:
env = "prod"

In [ ]:
exp = Explorer(env=env)

## List available tags

`.tags` returns the unique tag values in the current search context.

In [ ]:
ensembles = exp.filter(cls="ensemble")
tags = ensembles.tags
tags

## Ensembles with a given tag

Pass a single value to `tag=` to match ensembles that have that tag. The tags of an ensemble are in its metadata.

In [ ]:
tag = tags[0]
tagged = exp.filter(cls="ensemble", tag=tag)
print(f"Ensembles tagged '{tag}': {len(tagged)}")
[(ens.casename, ens.name, ens.get_property("tags")) for ens in tagged]

## Ensembles with any of several tags (OR)

Passing a list matches ensembles that have *at least one* of the tags.

In [ ]:
any_of = exp.filter(cls="ensemble", tag=tags[:2])
print(f"Ensembles tagged with any of {tags[:2]}: {len(any_of)}")
[(ens.casename, ens.name, ens.get_property("tags")) for ens in any_of]

## Ensembles with all of several tags (AND)

Chain `.filter()` calls to require every tag.

In [ ]:
all_of = exp.filter(cls="ensemble")
for t in tags[:2]:
    all_of = all_of.filter(tag=t)
print(f"Ensembles tagged with all of {tags[:2]}: {len(all_of)}")
[(ens.casename, ens.name, ens.get_property("tags")) for ens in all_of]

## Ensembles with or without tags

`tag=True` matches ensembles that have any tag, `tag=False` those that have none.

In [ ]:
with_tags = exp.filter(cls="ensemble", tag=True)
without_tags = exp.filter(cls="ensemble", tag=False)
print(f"Ensembles with tags:    {len(with_tags)}")
print(f"Ensembles without tags: {len(without_tags)}")

## Combine tags with other filters

`tag=` works together with the other filters, e.g. `asset`, `user` or `ensemble` (name). Narrowing down the selection also narrows down `.tags`.

In [ ]:
asset = tagged[0].asset
in_asset = exp.filter(cls="ensemble", asset=asset)
print(f"Tags used in asset {asset}: {in_asset.tags}")
print(f"Ensembles in {asset} tagged '{tag}': {len(in_asset.filter(tag=tag))}")

In [ ]:
# Cases that contain at least one ensemble with the tag
[(case.name, case.uuid) for case in tagged.cases]

## From tagged ensembles to data

An ensemble is itself a search context, so its data can be browsed directly.

In [ ]:
ens = tagged[0]
print(ens)
print(f"Surfaces: {len(ens.surfaces)}")
print(f"Tables:   {len(ens.tables)}")
print(f"Tables, tagnames: {ens.tables.tagnames}")

To search data across *all* ensembles with a tag, collect their uuids and filter on `fmu.ensemble.uuid`.

In [ ]:
ensemble_uuids = tagged.get_field_values("fmu.ensemble.uuid.keyword")
tables = exp.filter(
    cls="table",
    complex={"terms": {"fmu.ensemble.uuid.keyword": ensemble_uuids}},
)
print(f"Tables in ensembles tagged '{tag}': {len(tables)}")
tables.names